A1


In [8]:
import praw
import json
from redditClient import redditClient
import time
from datetime import datetime

In [9]:
# construct Reddit client
client = redditClient()

# sanity check, you should see your own username printed out
print(client.user.me())

Standard-Feeling5549


In [13]:
# Define multiple search queries
search_queries = ["best countries to visit", "top countries to visit"]

# Container for all search results
all_results = {}

for query in search_queries:
    print(f"Searching for: '{query}'")
    results = list(client.subreddit("all").search(
        query=query,
        sort="relevance",
        limit=500
    ))
    print(f"  Found {len(results)} posts.")
    
    # Add to dictionary using post ID to avoid duplicates
    for post in results:
        all_results[post.id] = post

# Final deduplicated list
unique_posts = list(all_results.values())

print(f"\nTotal unique posts across all queries: {len(unique_posts)}")


Searching for: 'best countries to visit'
  Found 231 posts.
Searching for: 'top countries to visit'
  Found 242 posts.

Total unique posts across all queries: 462


In [15]:
# Function to extract post + comments
def extract_submission_data(submission):
    submission.comments.replace_more(limit=0)
    return {
        "id": submission.id,
        "title": submission.title,
        "selftext": submission.selftext,
        "subreddit": submission.subreddit.display_name,
        "author": str(submission.author),
        "created_utc": submission.created_utc,
        "score": submission.score,
        "num_comments": submission.num_comments,
        "comments": [
            {
                "author": str(comment.author),
                "body": comment.body,
                "created_utc": comment.created_utc
            }
            for comment in submission.comments.list()
            if comment.body
        ]
    }

In [17]:
# Extract structured data
data = {"submissions": [extract_submission_data(post) for post in unique_posts]}

# Save to JSON
with open("best_countries_to_visit_results.json", "w") as f:
    json.dump(data, f, indent=2)


In [18]:
# Summary
print(f"\nSaved {len(data['submissions'])} posts to 'best_countries_to_visit_results.json'")



Saved 462 posts to 'best_countries_to_visit_results.json'
